# 自定义层、损失与自动求导

这份 notebook 讲的是 PyTorch 最有力量的一部分：框架不仅能调用现成组件，还允许我们自己定义损失函数、网络层和优化过程。

如果把深度学习看成“定义目标函数，然后不断优化参数”的过程，那么这份内容正好覆盖三块核心：
- 目标函数怎么写。
- 线性层怎么自己实现。
- 梯度是怎么被算出来、又怎么推动参数更新的。


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor, Normalize
from matplotlib import pyplot as plt
from tqdm import tqdm
from torch import nn
from torch.nn import functional as F

import torch
import numpy as np
import pandas as pd
import os

plt.rcParams['font.family'] = 'SimHei'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. 自定义损失函数的意义

损失函数本质上是在回答一个问题：模型现在“错得有多严重”。

这里用均方误差 MSE 作为例子：
\[
MSE = \frac{1}{n} \sum_{i=1}^{n}(\hat{y}_i - y_i)^2
\]

它适合回归任务，因为它会惩罚预测值和真实值之间的距离，而且大误差会被平方放大。


In [ ]:
# 自定义均方误差损失函数
def mse_loss(pred, target):
    return ((pred - target) ** 2).mean()

In [8]:
y_true = torch.tensor([3, -0.5, 2, 7], dtype=torch.float32)
y_pred = torch.tensor([2.5, 0.0, 2, 8], dtype=torch.float32)

loss = F.mse_loss(y_pred, y_true)  # 使用PyTorch内置的均方误差损失函数
print(loss)
custom_loss = mse_loss(y_pred, y_true)  # 使用自定义的均方误差损失函数
print(custom_loss)

tensor(0.3750)
tensor(0.3750)


## 2. 为什么先准备一个回归数据集

后面自定义层和训练相关的内容，需要有真实输入输出才能看出效果。这里选加州房价数据集，是因为它是典型的结构化回归任务，便于观察模型是否真的学到了数值映射关系。


In [10]:
housing = fetch_california_housing(data_home='../data')

x_train_, x_test, y_train_, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_, y_train_, test_size=0.25, random_state=42
)

dataset_maps = {
    'train': (x_train, y_train),
    'val': (x_val, y_val),
    'test': (x_test, y_test)
}

std = StandardScaler()
std.fit(x_train)


class HousingDataset(Dataset):
    def __init__(self, dataset_type='train'):
        self.x, self.y = dataset_maps[dataset_type]
        self.x = std.transform(self.x)
        self.x = torch.tensor(self.x, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


batch_size = 256

train_ds = HousingDataset('train')
val_ds = HousingDataset('val')
test_ds = HousingDataset('test')

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)

## 3. 自定义线性层在做什么

全连接层本质上就是一个线性变换：
\[
y = xW + b
\]

这里自己实现 `CustomizedLinear`，目的不是替代 `nn.Linear`，而是让你真正看清楚“参数”到底是什么：
- `W` 是权重矩阵，决定不同输入特征如何组合。
- `b` 是偏置，给模型一个整体平移能力。


In [11]:
# 自定义全连接层
class CustomizedLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(in_features, out_features))
        self.weights = nn.init.xavier_normal_(self.weights)
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        return x @ self.weights + self.bias

## 4. 数值求导为什么值得学

自动求导很方便，但如果没有导数直觉，训练过程就会像黑盒。这里用有限差分近似导数：
\[
f'(x) \approx \frac{f(x+h)-f(x-h)}{2h}
\]

它帮助我们从“变化率”角度理解梯度，而不是只记住 `backward()` 这个接口。


In [29]:
# 近似求导
def f(x, y=None):
    if y is None:
        return 3 * x ** 2 + 2 * x - 1
    return (x + 5) * y ** 2


def approximate_derivative(func, x, h=1e-6):
    return (func(x + h) - func(x - h)) / (2 * h)


print((f(1.0), approximate_derivative(f, 1.0)))

(4.0, 7.999999999785956)


In [30]:
# 偏导数

def approximate_gradient(func, x, y, h=1e-6):
    df_dx = approximate_derivative(lambda x: func(x, y), x, h)
    df_dy = approximate_derivative(lambda y: func(x, y), y, h)
    return df_dx, df_dy


print((f(2.0, 3.0), approximate_gradient(f, 2.0, 3.0)))

(63.0, (8.999999998593466, 42.00000000409432))


## 5. 自动求导与反向传播

PyTorch 会在前向计算时构建计算图，随后在 `backward()` 时按照链式法则自动计算梯度。核心思想是：
- 前向传播得到输出和损失。
- 反向传播把误差信号一层层传回去。
- 每个参数都能得到“往哪个方向改会更好”的梯度信息。


In [31]:
# torch 近似求导
# requires_grad=True 使得张量在计算过程中会构建计算图，从而可以通过反向传播计算梯度。
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
y = f(x1, x2)

# retain_graph=True 允许在计算梯度后继续使用计算图，这对于需要多次计算梯度的情况非常有用。
# 如果不设置这个参数，计算一次梯度后计算图就会被释放，无法再次使用。
dy_dx1, dy_dx2 = torch.autograd.grad(y, (x1, x2), retain_graph=True)
print((y, dy_dx1, dy_dx2))

(tensor(63., grad_fn=<MulBackward0>), tensor(9.), tensor(42.))


In [32]:
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
y = f(x1, x2)
y.backward()  # 计算梯度 同时计算 x1 和 x2 的梯度保存在 x1.grad 和 x2.grad 中
print(y, x1.grad, x2.grad)

tensor(63., grad_fn=<MulBackward0>) tensor(9.) tensor(42.)


In [33]:
# 高阶导数
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
y = f(x1, x2)

# create_graph=True 允许在计算梯度时构建计算图，这对于需要计算高阶导数的情况非常有用。
dy_dx1, dy_dx2 = torch.autograd.grad(y, (x1, x2), create_graph=True)
# allow_unused=True 允许在计算梯度时忽略未使用的输入，这对于某些情况下可能不需要计算所有输入的梯度非常有用。
d2y_dx1_dx1, d2y_dx1_dx2 = torch.autograd.grad(
    dy_dx1, (x1, x2), allow_unused=True)
d2y_dx2_dx1, d2y_dx2_dx2 = torch.autograd.grad(
    dy_dx2, (x1, x2), allow_unused=True)
print(d2y_dx1_dx1, d2y_dx1_dx2, d2y_dx2_dx1, d2y_dx2_dx2)

None tensor(6.) tensor(6.) tensor(14.)


## 6. 从梯度到优化

梯度本身不会自动让参数变好，真正更新参数的是优化步骤。最基础的 SGD 更新公式是：
\[
\theta \leftarrow \theta - \eta \nabla J(\theta)
\]

其中：
- `\theta` 是参数。
- `\eta` 是学习率。
- `\nabla J(\theta)` 是损失对参数的梯度。

这也是后面几乎所有训练循环的基本骨架。


In [40]:
# 模拟SGD优化过程
learning_rate = 0.3
x = torch.tensor(2.0, requires_grad=True)
for _ in range(100):
    z = f(x)
    z.backward()
    x.data -= learning_rate * x.grad.data  # 更新参数
    x.grad.zero_()  # 清零梯度以避免累积
print(x)

tensor(-0.3333, requires_grad=True)


In [41]:
# 结合optimizer
learning_rate = 0.01
x = torch.tensor(2.0, requires_grad=True)
optimizer = torch.optim.SGD([x], lr=learning_rate)
for _ in range(500):
    z = f(x)
    optimizer.zero_grad()  # 清零梯度
    z.backward()  # 计算梯度
    optimizer.step()  # 更新参数
    # optimizer.zero_grad()  # 清零梯度
print(x)

tensor(-0.3333, requires_grad=True)
